# STC Jawwy

In [17]:
"""
Here we install libraries that are not installed by default 
Example:  pyslsb
Feel free to add any library you are planning to use.
"""
!pip install pyxlsb


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# Import the required libraries 
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd     # provides high-performance, easy to use structures and data analysis tools
import pyxlsb           # Excel extention to read xlsb files (the input file)
import numpy as np      # provides fast mathematical computation on arrays and matrices

# Jawwy dataset
The dataset consists of details about each customer and the movies and/or tv shows watched in addition to the genre. 

You are required to work on task three to build a recommendation engine for our platform to Recommend movies to usesrs that they might be interested in¶


In [19]:
dataframe = pd.read_excel("stc TV Data Set_T3.xlsx", index_col=0)
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [20]:
# check the data shape
dataframe.shape

(1048575, 5)

In [21]:
# display the first 5 rows 
dataframe.head()

,user_id_maped,program_name,rating,date_,program_genre
0,26138,100 treets,1,2017-05-27,Drama
1,7946,Moana,1,2017-05-21,Animation
2,7418,The Mermaid Princess,1,2017-08-10,Animation
3,19307,The Mermaid Princess,2,2017-07-26,Animation
4,15860,Churchill,2,2017-07-07,Biography


In [22]:
# describe the numeric values in the dataset
dataframe.describe()

,user_id_maped,rating,date_
count,1.048575e+06,1.048575e+06,1048575
mean,1.709266e+04,2.497283e+00,2017-10-04 00:23:20.346183936
min,1.000000e+00,1.000000e+00,2017-03-14 00:00:00
25%,8.253000e+03,1.000000e+00,2017-06-10 00:00:00
50%,1.714900e+04,2.000000e+00,2017-10-14 00:00:00
75%,2.566500e+04,3.000000e+00,2018-01-21 00:00:00
max,3.428000e+04,4.000000e+00,2018-04-30 00:00:00
std,1.003513e+04,1.119837e+00,NaN


In [23]:
# check if any column has null value in the dataset
dataframe.isnull().any()

user_id_maped    False
program_name     False
rating           False
date_            False
program_genre    False
dtype: bool

In [24]:
# we import Visualization libraries 
# you can ignore and use any other graphing libraries 
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import plotly #a graphing library makes interactive, publication-quality graphs. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [25]:
"""
TODO build your Recommender system to Highlight Programs that users might be interested in
--------------------------------------------------------------------------------------
Approach: Collaborative Filtering (users who share the same taste watch the same programs)
"""
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

# 1) The raw file has one row per viewing event -> group it into one row per (user, program)
inter = (dataframe.groupby(['user_id_maped', 'program_name'])
                  .agg(watch_count=('rating', 'size'), avg_rating=('rating', 'mean'))
                  .reset_index())

# 2) Drop users / programs with less than 5 interactions (cold start = no signal)
MIN_INTERACTIONS = 5
inter = inter[inter.groupby('program_name')['user_id_maped'].transform('size') >= MIN_INTERACTIONS]
inter = inter[inter.groupby('user_id_maped')['program_name'].transform('size') >= MIN_INTERACTIONS]

user_list = np.sort(inter['user_id_maped'].unique())
item_list = np.sort(inter['program_name'].unique())
user_to_ix = {u: i for i, u in enumerate(user_list)}
item_to_ix = {p: i for i, p in enumerate(item_list)}
inter['u'] = inter['user_id_maped'].map(user_to_ix)
inter['i'] = inter['program_name'].map(item_to_ix)

# 3) Build the User-Item matrix (1 = the user watched the program)
R = csr_matrix((np.ones(len(inter), dtype=np.float32), (inter['u'].values, inter['i'].values)),
               shape=(len(user_list), len(item_list)))
print('Users:', R.shape[0], '| Programs:', R.shape[1], '| Interactions:', R.nnz)

# 4) USER-BASED CF : cosine similarity between users, keep the 50 nearest neighbours
K_NEIGHBOURS = 50
Un = normalize(R, axis=1)
user_sim = (Un @ Un.T).toarray()
np.fill_diagonal(user_sim, 0)
thr = np.partition(user_sim, -K_NEIGHBOURS, axis=1)[:, -K_NEIGHBOURS][:, None]
user_sim[user_sim < thr] = 0

# 5) ITEM-BASED CF : cosine similarity between programs ("watched X -> also watched Y")
In_ = normalize(R, axis=0)
item_sim = (In_.T @ In_).toarray()
np.fill_diagonal(item_sim, 0)

program_genre = dataframe.groupby('program_name')['program_genre'].agg(lambda s: s.mode()[0])
popularity = np.asarray(R.sum(axis=0)).ravel()


def recommend_user_based(user_id, n=5):
    """Recommend n programs using the n most similar users."""
    u = user_to_ix[user_id]
    scores = np.asarray(user_sim[u] @ R).ravel()
    scores[R[u].indices] = -np.inf                 # skip what he already watched
    top = np.argsort(-scores)[:n]
    return pd.DataFrame({'rank': range(1, n + 1),
                         'program_name': item_list[top],
                         'genre': [program_genre[item_list[j]] for j in top],
                         'score': np.round(scores[top], 3)})


def recommend_item_based(user_id, n=5):
    """Recommend n programs from the programs the user already watched."""
    u = user_to_ix[user_id]
    scores = np.asarray(R[u] @ item_sim).ravel()
    scores[R[u].indices] = -np.inf
    top = np.argsort(-scores)[:n]
    return pd.DataFrame({'rank': range(1, n + 1),
                         'program_name': item_list[top],
                         'genre': [program_genre[item_list[j]] for j in top],
                         'score': np.round(scores[top], 3)})


def similar_programs(program_name, n=5):
    """The n programs most watched by the same audience as `program_name`."""
    i = item_to_ix[program_name]
    top = np.argsort(-item_sim[i])[:n]
    return pd.DataFrame({'rank': range(1, n + 1),
                         'program_name': item_list[top],
                         'genre': [program_genre[item_list[j]] for j in top],
                         'similarity': np.round(item_sim[i][top], 4),
                         'watchers': popularity[top].astype(int)})


print('\nExample - recommendations for user', user_list[100])
print(recommend_user_based(user_list[100], 5).to_string(index=False))

Users: 6700 | Programs: 6927 | Interactions: 428022

Example - recommendations for user 451
 rank                          program_name     genre  score
    1                                Trolls Animation  6.656
    2 The Jetsons & WWE: Robo-WrestleMania! Animation  6.008
    3                  The Mermaid Princess Animation  5.864
    4                                 Rings    Horror  5.130
    5                             Ferdinand Animation  5.072


In [26]:
"""
TODO show the recommendations (top 5) for the people who watched "Moana" movie
"""
# A) The 5 programs most watched by the same users who watched Moana
print('People who watched "Moana" also watched:')
display(similar_programs('Moana', 5))

# B) Top 5 for the whole group of Moana watchers
moana_ix = item_to_ix['Moana']
moana_users = np.asarray(R[:, moana_ix].todense()).ravel() > 0
print('Users who watched Moana:', moana_users.sum())

group_scores = np.asarray((R[moana_users] @ item_sim).sum(axis=0)).ravel()
group_scores[moana_ix] = -np.inf                  # do not recommend Moana itself
top5 = np.argsort(-group_scores)[:5]

display(pd.DataFrame({'rank': range(1, 6),
                      'program_name': item_list[top5],
                      'genre': [program_genre[item_list[j]] for j in top5],
                      'group_score': np.round(group_scores[top5], 1),
                      'watched_by_moana_fans': np.asarray(R[moana_users].sum(axis=0)).ravel()[top5].astype(int)}))

# C) Personalised sample for 3 real Moana watchers
for uid in user_list[moana_users][[5, 50, 200]]:
    u = user_to_ix[uid]
    print('\n' + '=' * 70)
    print('USER', uid, '|', R[u].nnz, 'programs watched')
    print('History (sample):', ', '.join([item_list[i] for i in R[u].indices][:6]))
    print(recommend_item_based(uid, 5).to_string(index=False))

People who watched "Moana" also watched:


,rank,program_name,genre,similarity,watchers
0,1,Trolls,Animation,0.6386,2180
1,2,Surf's Up : WaveMania,Animation,0.6048,1321
2,3,The Mermaid Princess,Animation,0.5585,1901
3,4,The Jetsons & WWE: Robo-WrestleMania!,Animation,0.5180,1430
4,5,The Boss Baby,Animation,0.5105,2745


Users who watched Moana: 1817


,rank,program_name,genre,group_score,watched_by_moana_fans
0,1,The murfs,Animation,24204.000000,336
1,2,The Boss Baby,Animation,24105.599609,1140
2,3,Cloudy With a Chance of Meatballs,Animation,23643.800781,260
3,4,The Amazing pider-Man,Action,23120.900391,319
4,5,Bean,Comedy,22832.000000,243



USER 205 | 125 programs watched
History (sample): 22 Jump treet, 24 Carat  (T), Akoon Aw La    Ep01, Alien: Covenant, Aliens vs. Predator: Requiem, Alvin and the Chipmunks: The Road Chip
 rank                            program_name     genre     score
    1                               The murfs Animation 19.959000
    2 Howard Lovecraft and the Frozen Kingdom Animation 19.889000
    3                      The Little Vampire Animation 18.768999
    4                 Alvin and the Chipmunks Animation 18.670000
    5                                    Cars Animation 18.653000

USER 1000 | 47 programs watched
History (sample): 100 treets, 22 Jump treet, Alexander and the Terrible  Horrible  No G, Alvin and the Chipmunks, An Inconvenient equel: Truth to Power, Assassin's Creed
 rank                          program_name     genre  score
    1                         The Boss Baby Animation 10.555
    2                  The Mermaid Princess Animation 10.157
    3                 The Birt